# App 0 · 环境自检 — 5 分钟避免后面 30 分钟踩坑

跑这一节之前先跑这一节。听起来废话，但这门课走过的反馈数据告诉我们一件事：**80% 的"卡住"问题不在内容本身，而在环境**。最常见的几种——

- 没装可选 SDK，跑到 App5 才发现 mcp 包缺，反复重装
- DashScope / OpenAI API key 没配，跑到 App2 才看见 401 错误
- Python 版本太老，sentence-transformers 装不上
- 中文字体没装，matplotlib 把汉字画成方块

这一节的任务是把这些问题**前置到 5 分钟内排查清楚**。下面 6 个 step 走一遍，最后 `verify_environment()` 给你一个明确判断："可以去 App1 了"或"先修 X 再来"。

> 这一节不依赖任何前置 App，是整门课的第一节。所有 App 共用同一份 backend 抽象（DashScope / OpenAI / Ollama 任选），所以 Step 2 的 `utils.config.setup()` 跑通就等于全栈都能跑——后面任何一节有"为什么 LLM 不响应"的问题，都先回这里看是不是 setup 出错了。

---

## Step 1 · Python 版本 + `utils/` 可 import

把 repo 根目录加到 `sys.path`，让 `import utils.xxx` 在 `Applications/` 子目录里也能用。
所有 App5–8 都用同一段 `_root` 探测代码，App0 复用它。


In [1]:
import os, sys

_status: dict[str, tuple[str, str]] = {}  # 收集每步结果，最后 verify_environment 用

# ── 自动定位 repo 根目录 ───────────────────────────────────────
_cur = os.path.abspath("")
_root = None
for _c in [_cur, os.path.dirname(_cur), os.path.dirname(os.path.dirname(_cur))]:
    if os.path.isdir(os.path.join(_c, "utils")) and os.path.isfile(os.path.join(_c, "README.md")):
        _root = _c
        break
if _root is None:
    raise RuntimeError("找不到 repo 根目录（应包含 utils/ 与 README.md）")
os.chdir(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
print(f"📂 repo root: {_root}")

# ── Python 版本 ────────────────────────────────────────────
print(f"🐍 Python: {sys.version.split()[0]}  ({sys.executable})")
if sys.version_info >= (3, 9):
    _status["python"] = ("ok", f"Python {sys.version_info.major}.{sys.version_info.minor}")
    print("  ✅ Python ≥ 3.9")
else:
    _status["python"] = ("fail", f"Python {sys.version_info.major}.{sys.version_info.minor} < 3.9")
    print("  ❌ Python < 3.9 — 部分依赖（如 langchain 系）会装不上，请升级")

# ── utils 可导入 ───────────────────────────────────────────
try:
    import utils  # noqa: F401
    import utils.config  # noqa: F401
    _status["utils_import"] = ("ok", "utils.config import 成功")
    print("  ✅ utils.config import 成功")
except Exception as exc:  # 任何 ImportError / SyntaxError 都要兜住
    _status["utils_import"] = ("fail", f"{type(exc).__name__}: {exc}")
    print(f"  ❌ utils import 失败：{exc}")


📂 repo root: C:\Users\lvbab\Documents\GitHub\LLM-Agent-Core_Concept_Code
🐍 Python: 3.9.25  (E:\conda\envs\llmc\python.exe)
  ✅ Python ≥ 3.9
  ✅ utils.config import 成功


---

## Step 2 · 装 LLM / Embedding 后端

`utils.config.setup()` 会读 `.env`，挑出激活的 LLM/Embedding 后端（OpenAI / DashScope / Ollama / HF / vLLM）。
**不会**真发请求——只是把 backend 工厂构建好。看到打印的 backend 名字就算通过。


In [2]:
try:
    from utils.config import setup
    env = setup()
    llm_name = getattr(env, "llm_backend", None) or os.environ.get("LLM_BACKEND", "?")
    emb_name = getattr(env, "embedding_backend", None) or os.environ.get("EMBEDDING_BACKEND", "?")
    print(f"  ✅ utils.config.setup() ok")
    print(f"     LLM backend       → {llm_name}")
    print(f"     Embedding backend → {emb_name}")
    _status["config_setup"] = ("ok", f"llm={llm_name}, emb={emb_name}")
except FileNotFoundError as exc:
    _status["config_setup"] = ("warn", "缺 .env 文件")
    print(f"  ⚠️ 没找到 .env：{exc}")
    print("     复制根目录 `.env.example` 为 `.env` 后填 API Key 再重跑")
except Exception as exc:
    _status["config_setup"] = ("fail", f"{type(exc).__name__}: {exc}")
    print(f"  ❌ setup() 失败：{exc}")


[OK] 使用系统环境变量中的 DASHSCOPE_API_KEY
课程环境配置:
  API Key:   ✓ 已配置
  LLM:       dashscope / qwen-plus
  Embedding: dashscope / text-embedding-v3
  ✅ utils.config.setup() ok
     LLM backend       → dashscope
     Embedding backend → dashscope


---

## Step 3 · 核心深度依赖（Apps 1–4 必需）

PyTorch / NumPy / Transformers / SentenceTransformers 是 Apps 1–4 的最小集合。
任何一个 ❌ 都意味着对应 App 跑不动；按提示 `pip install` 即可。


In [3]:
_core_deps = {
    "numpy":                "数值计算",
    "torch":                "PyTorch",
    "transformers":         "HuggingFace Transformers",
    "sentence_transformers": "SentenceTransformers (Embedding & Reranker)",
    "openai":               "OpenAI SDK（DashScope 兼容）",
    "tiktoken":             "Tiktoken Tokenizer",
}
_core_missing = []
for pkg, desc in _core_deps.items():
    try:
        mod = __import__(pkg)
        ver = getattr(mod, "__version__", "?")
        print(f"  ✅ {pkg:24s} {ver:12s}  {desc}")
    except ImportError:
        print(f"  ❌ {pkg:24s} —            {desc}  →  pip install {pkg.replace('_', '-')}")
        _core_missing.append(pkg)

# GPU 提示（非阻断）
try:
    import torch
    if torch.cuda.is_available():
        print(f"  🎯 CUDA 可用：{torch.cuda.get_device_name(0)}")
    else:
        print("  ⚠️ 无 GPU；Apps 1–4 在 CPU 也能跑（推理慢一点）")
except Exception:
    pass

_status["core_deps"] = ("ok", "all-installed") if not _core_missing \
    else ("fail", f"缺 {len(_core_missing)} 个: {_core_missing}")


  ✅ numpy                    2.0.2         数值计算


  ✅ torch                    2.8.0+cu128   PyTorch


  ✅ transformers             4.57.3        HuggingFace Transformers


  ✅ sentence_transformers    5.1.2         SentenceTransformers (Embedding & Reranker)


  ✅ openai                   2.14.0        OpenAI SDK（DashScope 兼容）
  ✅ tiktoken                 0.12.0        Tiktoken Tokenizer
  🎯 CUDA 可用：NVIDIA GeForce RTX 5080


---

## Step 4 · Apps 5–8 可选 SDK

这些 SDK **不装也能跑** —— `utils/{mcp_helpers,observability,skills_helpers}.py` 都内置了 mock fallback。
但装上 SDK 后这几个 App 会有更接近生产的体验（真 stdio JSON-RPC、真 Langfuse trace 上传、真 BM25 hybrid）。

只跑 Apps 1–4 → 整段全 ⚠️ 完全可以接受。
要跑 Apps 5–8 → 建议至少装 `mcp` + `langfuse` + `rank-bm25`。


In [4]:
_optional_deps = {
    "mcp":                 ("App5 MCP Server",       "pip install 'mcp>=0.9'"),
    "langfuse":            ("App7 LLMOps trace",     "pip install 'langfuse>=2.0'"),
    "rank_bm25":           ("App2 Hybrid RAG",       "pip install rank-bm25"),
    "anthropic":           ("App6 Skills (实 LLM)",   "pip install 'anthropic>=0.30'"),
    "opentelemetry":       ("OTEL 标准化 trace",      "pip install opentelemetry-api opentelemetry-sdk"),
}
_optional_missing = []
for pkg, (purpose, install_cmd) in _optional_deps.items():
    try:
        __import__(pkg)
        print(f"  ✅ {pkg:18s} ({purpose})")
    except ImportError:
        print(f"  ⚠️ {pkg:18s} 未装 → {purpose} 走 mock fallback")
        print(f"     需要时: {install_cmd}")
        _optional_missing.append(pkg)

if not _optional_missing:
    _status["optional_deps"] = ("ok", "全部 SDK 就位")
else:
    _status["optional_deps"] = ("warn", f"{len(_optional_missing)}/{len(_optional_deps)} 走 mock fallback")


  ⚠️ mcp                未装 → App5 MCP Server 走 mock fallback
     需要时: pip install 'mcp>=0.9'
  ⚠️ langfuse           未装 → App7 LLMOps trace 走 mock fallback
     需要时: pip install 'langfuse>=2.0'
  ⚠️ rank_bm25          未装 → App2 Hybrid RAG 走 mock fallback
     需要时: pip install rank-bm25
  ⚠️ anthropic          未装 → App6 Skills (实 LLM) 走 mock fallback
     需要时: pip install 'anthropic>=0.30'
  ⚠️ opentelemetry      未装 → OTEL 标准化 trace 走 mock fallback
     需要时: pip install opentelemetry-api opentelemetry-sdk


---

## Step 5 · 数据资产

`data/`（语料、向量索引）、`fonts/`（matplotlib 中文字体）、`docs/architecture.md`（全景图）。
缺字体只影响图表的中文渲染；缺 `data/` 部分文件只影响特定章节，不阻断 App1–4。


In [5]:
from pathlib import Path

_assets = {
    "data/":                       (True,  "训练 / 语料数据"),
    "fonts/":                      (False, "中文字体"),
    "utils/":                      (True,  "共享后端模块"),
    "docs/architecture.md":        (False, "架构全景图"),
    "Applications/":               (True,  "App 案例库"),
}
_asset_fail = []
for path_str, (required, desc) in _assets.items():
    p = Path(_root) / path_str
    if p.exists():
        kind = "📁" if p.is_dir() else "📄"
        print(f"  ✅ {kind} {path_str:32s} {desc}")
    else:
        mark = "❌" if required else "⚠️"
        print(f"  {mark} {path_str:32s} 不存在 — {desc} ({'必需' if required else '可选'})")
        if required:
            _asset_fail.append(path_str)

_status["assets"] = ("ok", "全部就位") if not _asset_fail \
    else ("fail", f"缺必需资产: {_asset_fail}")


  ✅ 📁 data/                            训练 / 语料数据
  ⚠️ fonts/                           不存在 — 中文字体 (可选)
  ✅ 📁 utils/                           共享后端模块
  ✅ 📄 docs/architecture.md             架构全景图
  ✅ 📁 Applications/                    App 案例库


---

## Step 6 · 综合判断（可以去 App1 了吗？）

`verify_environment()` 把上面 5 步的 `_status` 收齐，用一句话告诉你下一步该干什么。


In [6]:
def verify_environment(status: dict) -> bool:
    """汇总所有 step 的状态，打印结论 + 下一步建议。返回 True 表示可以进 App1。"""
    print("=" * 60)
    print("环境自检综合结果")
    print("=" * 60)
    icons = {"ok": "✅", "warn": "⚠️", "fail": "❌"}
    required_keys = ("python", "utils_import", "config_setup", "core_deps", "assets")
    optional_keys = ("optional_deps",)

    for k in required_keys + optional_keys:
        if k not in status:
            print(f"  ⏭ {k:18s} 未跑（请按顺序跑前面的 cell）")
            continue
        level, detail = status[k]
        print(f"  {icons[level]} {k:18s} {detail}")
    print()

    # 必需项必须全 ok 才放行
    fails = [k for k in required_keys if status.get(k, ("fail",))[0] == "fail"]
    if fails:
        print(f"❌ {len(fails)} 项必需检查失败：{fails}")
        print("   按上面的提示修复后重新 Run All。")
        return False

    warns = [k for k in optional_keys if status.get(k, ("ok",))[0] == "warn"]
    if warns:
        print("⚠️ 部分可选 SDK 未装，Apps 5–8 会走 mock fallback。")
        print("   只跑 Apps 1–4 的话可以直接去 → Applications/App1_ReAct_Agent.ipynb")
    else:
        print("🚀 全部就位！下一步：Applications/App1_ReAct_Agent.ipynb")
    return True


verify_environment(_status)


环境自检综合结果
  ✅ python             Python 3.9
  ✅ utils_import       utils.config import 成功
  ✅ config_setup       llm=dashscope, emb=dashscope
  ✅ core_deps          all-installed
  ✅ assets             全部就位
  ⚠️ optional_deps      5/5 走 mock fallback

⚠️ 部分可选 SDK 未装，Apps 5–8 会走 mock fallback。
   只跑 Apps 1–4 的话可以直接去 → Applications/App1_ReAct_Agent.ipynb


True